<a href="https://colab.research.google.com/github/aa4758/ML_DL_study/blob/master/Self%20Study%20ML%20%2B%20DL/Ensemble_Learining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Data Type; structured data: data formed by structure such as csv, database, or excel
# Data Type; unstructured data: data doesn't have any structure such as image, text data, music, etc
# NoSQL database can storage text or json file but we're talking about general example

In [2]:
# Machine Learning algorithm fit to structured data well
# Most ineffective algorithm to handle structured data is ensumble learning which is based on DecisionTree

In [6]:
# Random Forest is one of the ensemble learning has stable performance
# Random Forest literaly makes a forest to make decision tree randomly and final prediction by prediction of each decision tree
# Before to train each tree, shuffle data without replacement
# Bootstrap Sample: sampling with replacement, sample size = training set size
# When we split each node, randomly use some features from all and find best split.
# RadomForestClassifier basically choose root of number of all features
# RandomForestRegressor use all features
# RandomForestClassifier basically train 100 decision tree in these ways
# In classification, find out a mean of each tree's class probability and set a class with highest mean as a prediction
# In Regressor, just mean each tree's predictions
# RandomForest use random sample and features ==> prevent overfitting to training set and able to get stable performance on validation set and test set

In [5]:
# split: split the node to left and right node by feature condition
# best split: split point that increase impurity most

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
wine = pd.read_csv('https://bit.ly/wine-date')
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size = 0.2, random_state = 42)

In [10]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier         # basically use 100 decision tree
rf = RandomForestClassifier(n_jobs=-1, random_state=42)     # recommand to set n_jobs as -1
scores = cross_validate(rf, train_input, train_target, return_train_score = True, n_jobs=-1)      # `return_train_score=True`: not only cross_validate returns a valdiation score also return a training score
print(np.mean(scores['train_score']), np.mean(scores['test_score']))                  # overfitting

0.9973541965122431 0.8905151032797809


In [11]:
# RandomTree has all hyperparameters of the DecisionTreeClassifier since it is ensemble of decision tree.
# RandomTree also calculate critical rate of features

In [12]:
rf.fit(train_input, train_target)
print(rf.feature_importances_)        # DecisionTreeClassifier() feature importcances: [0.123.. , 0.868.. , 0.007..]

[0.23167441 0.50039841 0.26792718]


In [14]:
# importances of sugar is decreased and ohter features are increased
# beacuse random forest choose some of the features randomly to train decision tree which means other features get more chances to contribute training model ==> decreased overfitting and increased general performance
# random forest train decision tree with bootstrap sample ==> there are some of samples weren't used to train and we can use those samples to validate decision tree called OOB (out of bag)

In [15]:
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)
rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


In [16]:
# Extra Trees is similar with Random Forest but it does not use bootstrap, use all training set
# Extra Trees does not find best split, randomly split ==> low performance but prevent overfitting and increase validation score since it ensemble more trees
# Extra Trees need to train more Decision Tree than Random Forest but also faster than Random Forest

In [18]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, n_jobs=-1, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


In [19]:
et.fit(train_input, train_target)
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


In [21]:
# Gradient Boosting: use low depth tree to complement loss of binary tree
# GradientBoostingClassifier use 100 of 3 depth decision tree ==> strong to overfitting and high general performance
# Gradient Boosting use gradient descent to add tree to ensemble ==> use logistic loss fn in classification and MAE in regression

In [23]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target, n_jobs=-1, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))        # not overfitting

0.8881086892152563 0.8720430147331015


In [24]:
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)             # increase n_estimator from 100 to 500 and learning_rate from 0.1 to 0.2
scores = cross_validate(gb, train_input, train_target, n_jobs=-1, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9464595437171814 0.8780082549788999


In [25]:
gb.fit(train_input, train_target)
print(gb.feature_importances_)      # bigger focus on sugar than random forest

[0.15887763 0.6799705  0.16115187]


In [26]:
# in general, gradient boosting has higher performance than randomforest but needs longer training time since it ensembles trees in order one by one ==> does not have n_jobs hyperparameter

In [27]:
# Histogram-based Gradient Boosting: devide input features by 256 intervals which makes faster to find best split
# ==> if we have million inputs than we have to check million inputs to find best split point from GB. but from HGB, we only need to check 256 points.
# HGB use one interval for null so we don't have to pre-processing null of input features
# HGB fix number of tree, use max_iter instead of n_estimator

In [29]:
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target, n_jobs=-1, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


In [31]:
from sklearn.inspection import permutation_importance         # calculate features importance by randomly shuffle features one by one and observate how performance is changed
                                                              # `n_repeats`: number of shuffle, default is 5
hgb.fit(train_input, train_target)
result = permutation_importance(hgb, train_input, train_target, n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)                                # permutaion_importance returns importances, importances_mean, importances_std

[0.08876275 0.23438522 0.08027708]


In [32]:
result = permutation_importance(hgb, test_input, test_target, n_repeats=10, random_state=42, n_jobs=-1)       # also worked for test set
print(result.importances_mean)

[0.05969231 0.20238462 0.049     ]


In [35]:
hgb.score(test_input, test_target)

0.8723076923076923

In [38]:
# Other libraries: xgboost, lightgbm

In [36]:
from xgboost import XGBClassifier
xgb = XGBClassifier(tree_method='hist', random_state=42)                                          # `tree_method='hist'`: HGB algorithm
scores = cross_validate(xgb, train_input, train_target, n_jobs=-1, return_train_score=True)       # able to use with cross validation
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9567059184812372 0.8783915747390243


In [37]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(random_state=42)
scores = cross_validate(lgb, train_input, train_target, n_jobs=-1, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.935828414851749 0.8801251203079884
